# Rarity Analysis for Reznar's Arcane Oddities

This notebook is the Part 2 analysis report. It uses the data already extracted and loaded into the local embedded Postgres database. It does not rerun the extraction pipeline, call OpenAI, or modify the database.

## 1. Purpose and framing

Rarity is supposed to roughly reflect the power, complexity, and specialness of a magic item. A common item should usually be simpler and less powerful than a rare, legendary, or artifact-level item.

The goal here is not to replace Reznar's judgment with a model. Instead, we use the structured ontology fields from Part 1 to find items whose extracted characteristics look unusual for their stated rarity tier. Those items become review candidates.

This is exploratory. The catalog is small, the extraction may contain minor errors, and rarity can depend on lore or campaign design choices that are not fully captured in structured fields.

## 2. Load the extracted dataset

The analysis reads from the local embedded Postgres database using `db.connect()`. The two tables used are:

- `magic_items`: one canonical row per item.
- `item_entities`: related ontology records such as effects, usage limits, charge pools, drawbacks, lore, and rarity variants.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from db import connect

plt.style.use("default")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 300)


def cursor_column_name(description_entry):
    name = getattr(description_entry, "name", None)
    if name is not None:
        return name
    return description_entry[0]


def read_sql_df(query, params=None):
    """Run a read-only SQL query through db.connect() and return a DataFrame."""
    with connect() as conn:
        cur = conn.execute(query, params or ())
        rows = cur.fetchall()
        columns = [cursor_column_name(col) for col in cur.description]
    return pd.DataFrame(rows, columns=columns)


magic_items = read_sql_df(
    """
    select
        id::text,
        name,
        header_line,
        item_category,
        item_form,
        subtype,
        rarity,
        wear_slot,
        attunement_required,
        is_cursed,
        is_consumable,
        source_pages,
        raw_text,
        needs_review
    from magic_items
    order by source_pages, name
    """
)

item_entities = read_sql_df(
    """
    select
        id::text,
        magic_item_id::text,
        entity_type,
        source_pages,
        data,
        needs_review
    from item_entities
    order by entity_type, id
    """
)

print(f"magic_items rows: {len(magic_items)}")
print(f"item_entities rows: {len(item_entities)}")

display(magic_items.head())
display(item_entities.head())

## 3. Basic data checks

Before modeling, we check whether the loaded data has the expected shape: a reasonable rarity distribution, related entity records, no obvious duplicate item names, and multi-page items represented as single records.

In [ ]:
rarity_order = ["common", "uncommon", "rare", "very rare", "legendary", "artifact"]


def as_list(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return list(value) if isinstance(value, tuple) else []


rarity_counts = magic_items["rarity"].replace("", np.nan).fillna("missing").value_counts()
category_counts = magic_items["item_category"].replace("", np.nan).fillna("missing").value_counts()
entity_type_counts = item_entities["entity_type"].replace("", np.nan).fillna("missing").value_counts() if not item_entities.empty else pd.Series(dtype=int)

missing_rarity_count = int(magic_items["rarity"].replace("", np.nan).isna().sum())

duplicate_names = magic_items["name"].replace("", np.nan).dropna().value_counts()
duplicate_names = duplicate_names[duplicate_names > 1]

magic_items_checks = magic_items.copy()
magic_items_checks["source_page_count"] = magic_items_checks["source_pages"].apply(lambda value: len(as_list(value)))
multi_page_items = magic_items_checks.loc[
    magic_items_checks["source_page_count"] > 1,
    ["name", "rarity", "item_category", "source_pages"],
]

attunement_by_rarity = (
    magic_items.assign(attunement_required=magic_items["attunement_required"].fillna(False).astype(bool))
    .groupby("rarity", dropna=False)["attunement_required"]
    .mean()
    .reindex(rarity_order)
    .dropna()
)

print(f"Missing rarity count: {missing_rarity_count}")
print(f"Duplicate item names: {len(duplicate_names)}")
print(f"Multi-page items: {len(multi_page_items)}")

display(pd.DataFrame({"rarity_count": rarity_counts}))
display(pd.DataFrame({"item_category_count": category_counts}))
display(pd.DataFrame({"entity_type_count": entity_type_counts}))
display(pd.DataFrame({"duplicate_count": duplicate_names}))
display(multi_page_items)

In [ ]:
ordered_rarity_counts = rarity_counts.reindex(rarity_order + ["missing"]).dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ordered_rarity_counts.plot(kind="bar", ax=axes[0], color="#4c78a8")
axes[0].set_title("Magic items by stated rarity")
axes[0].set_xlabel("Rarity")
axes[0].set_ylabel("Item count")

if len(entity_type_counts):
    entity_type_counts.sort_index().plot(kind="bar", ax=axes[1], color="#59a14f")
    axes[1].set_title("Related ontology entity records")
    axes[1].set_xlabel("Entity type")
    axes[1].set_ylabel("Record count")
else:
    axes[1].text(0.5, 0.5, "No related entities loaded", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

if len(attunement_by_rarity):
    fig, ax = plt.subplots(figsize=(7, 4))
    (attunement_by_rarity * 100).plot(kind="bar", ax=ax, color="#f28e2b")
    ax.set_title("Attunement requirement rate by rarity")
    ax.set_xlabel("Rarity")
    ax.set_ylabel("Items requiring attunement (%)")
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("No attunement-by-rarity chart shown because rarity values are missing.")

## 4. Feature engineering

The model needs one row per magic item. We start with stable item-level fields, then add counts from related ontology entities.

Important exclusions:

- `header_line` is not used as a model feature because it often contains the rarity directly.
- `rarity` is the target, not a feature.
- `is_artifact` is not used because it is too directly tied to artifact rarity.

The `complexity_score` below is a rough interpretive helper, not an official D&D pricing or rarity formula.

In [ ]:
def parse_jsonish(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}


def text_word_count(value):
    return len(str(value or "").split())


def text_char_count(value):
    return len(str(value or ""))


feature_df = magic_items.copy()
feature_df["attunement_required"] = feature_df["attunement_required"].fillna(False).astype(bool).astype(int)
feature_df["is_cursed"] = feature_df["is_cursed"].fillna(False).astype(bool).astype(int)
feature_df["is_consumable"] = feature_df["is_consumable"].fillna(False).astype(bool).astype(int)
feature_df["raw_text_word_count"] = feature_df["raw_text"].apply(text_word_count)
feature_df["raw_text_char_count"] = feature_df["raw_text"].apply(text_char_count)
feature_df["source_page_count"] = feature_df["source_pages"].apply(lambda value: len(as_list(value)))

base_columns = [
    "id",
    "name",
    "rarity",
    "item_category",
    "item_form",
    "subtype",
    "attunement_required",
    "is_cursed",
    "is_consumable",
    "raw_text_word_count",
    "raw_text_char_count",
    "source_page_count",
    "source_pages",
    "raw_text",
]
feature_df = feature_df[base_columns].copy()

for column in ["total_entity_count", "effect_count", "usage_limit_count", "charge_pool_count", "max_charge_pool", "total_charge_pools"]:
    feature_df[column] = 0

if not item_entities.empty:
    entity_features = item_entities.copy()
    entity_features["data_dict"] = entity_features["data"].apply(parse_jsonish)

    total_counts = entity_features.groupby("magic_item_id").size().rename("total_entity_count")
    feature_df = feature_df.drop(columns=["total_entity_count"]).merge(
        total_counts,
        left_on="id",
        right_index=True,
        how="left",
    )

    type_counts = pd.crosstab(entity_features["magic_item_id"], entity_features["entity_type"])
    entity_count_mapping = {
        "ItemEffect": "effect_count",
        "ItemUsageLimit": "usage_limit_count",
        "ItemChargePool": "charge_pool_count",
    }
    for entity_type, column in entity_count_mapping.items():
        values = type_counts[entity_type] if entity_type in type_counts.columns else pd.Series(0, index=type_counts.index)
        feature_df = feature_df.drop(columns=[column]).merge(
            values.rename(column),
            left_on="id",
            right_index=True,
            how="left",
        )

    effects = entity_features.loc[entity_features["entity_type"] == "ItemEffect"].copy()
    if not effects.empty:
        effects["effect_category"] = effects["data_dict"].apply(lambda data: data.get("category") or "unknown")
        effect_category_counts = pd.crosstab(effects["magic_item_id"], effects["effect_category"])
        effect_category_counts = effect_category_counts.add_prefix("effect_category_")
        feature_df = feature_df.merge(effect_category_counts, left_on="id", right_index=True, how="left")

    charge_pools = entity_features.loc[entity_features["entity_type"] == "ItemChargePool"].copy()
    if not charge_pools.empty:
        charge_pools["max_charges"] = pd.to_numeric(
            charge_pools["data_dict"].apply(lambda data: data.get("max_charges")),
            errors="coerce",
        )
        charge_summary = charge_pools.groupby("magic_item_id").agg(
            max_charge_pool=("max_charges", "max"),
            total_charge_pools=("id", "count"),
        )
        feature_df = feature_df.drop(columns=["max_charge_pool", "total_charge_pools"]).merge(
            charge_summary,
            left_on="id",
            right_index=True,
            how="left",
        )

numeric_fill_columns = feature_df.select_dtypes(include=["number"]).columns
feature_df[numeric_fill_columns] = feature_df[numeric_fill_columns].fillna(0)

feature_df["complexity_score"] = (
    feature_df["effect_count"]
    + 1.5 * feature_df["usage_limit_count"]
    + 2.0 * feature_df["charge_pool_count"]
    + 0.01 * feature_df["raw_text_word_count"]
    + 1.0 * feature_df["attunement_required"]
)

display(feature_df.head())
display(feature_df[["name", "rarity", "effect_count", "usage_limit_count", "charge_pool_count", "complexity_score"]].sort_values("complexity_score", ascending=False).head(10))

## 5. Modeling approach

Because the dataset is small, the model is intentionally simple. We compare a rarity prediction model against a baseline that always predicts the most frequent rarity. If the model only barely beats that baseline, the anomaly list should be treated with extra caution.

Note: The final extracted catalog does not contain common items, so the model learns distinctions among uncommon and higher rarity tiers only.

Metrics used below:

- Accuracy: the share of items whose rarity was predicted exactly.
- Macro F1: a class-balanced score that gives small rarity classes more weight than accuracy does.
- Confusion matrix: shows which rarity tiers are most often confused with each other.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rarity_rank = {rarity: index for index, rarity in enumerate(rarity_order)}

model_df = feature_df.dropna(subset=["rarity"]).copy()
model_df["rarity"] = model_df["rarity"].astype(str).str.strip().str.lower()
model_df = model_df[model_df["rarity"].isin(rarity_rank)].copy()
model_df["rarity_rank"] = model_df["rarity"].map(rarity_rank)

categorical_features = ["item_category", "item_form", "subtype"]
effect_category_features = [column for column in model_df.columns if column.startswith("effect_category_")]
numeric_features = [
    "attunement_required",
    "is_cursed",
    "is_consumable",
    "raw_text_word_count",
    "raw_text_char_count",
    "source_page_count",
    "total_entity_count",
    "effect_count",
    "usage_limit_count",
    "charge_pool_count",
    "max_charge_pool",
    "total_charge_pools",
    "complexity_score",
] + effect_category_features

model_df[categorical_features] = model_df[categorical_features].replace("", np.nan).fillna("unknown")
model_df[numeric_features] = model_df[numeric_features].fillna(0)

X = model_df[categorical_features + numeric_features]
y = model_df["rarity"]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", make_one_hot_encoder(), categorical_features),
        ("numeric", StandardScaler(), numeric_features),
    ]
)

baseline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]
)

rarity_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

class_counts = y.value_counts()
minimum_class_count = int(class_counts.min()) if not class_counts.empty else 0
labels = [rarity for rarity in rarity_order if rarity in set(y)]

if len(class_counts) > 1 and minimum_class_count >= 2:
    n_splits = min(5, minimum_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    baseline_pred = cross_val_predict(baseline, X, y, cv=cv)
    model_pred = cross_val_predict(rarity_model, X, y, cv=cv)
    evaluation_note = f"Used stratified {n_splits}-fold cross-validation."
else:
    baseline.fit(X, y)
    rarity_model.fit(X, y)
    baseline_pred = baseline.predict(X)
    model_pred = rarity_model.predict(X)
    evaluation_note = "At least one rarity class has fewer than two examples, so cross-validation is limited. These training-set metrics are descriptive only."

metrics = pd.DataFrame(
    [
        {
            "model": "Most frequent rarity baseline",
            "accuracy": accuracy_score(y, baseline_pred),
            "macro_f1": f1_score(y, baseline_pred, average="macro", zero_division=0),
        },
        {
            "model": "Balanced logistic regression",
            "accuracy": accuracy_score(y, model_pred),
            "macro_f1": f1_score(y, model_pred, average="macro", zero_division=0),
        },
    ]
)

cm = confusion_matrix(y, model_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"actual: {label}" for label in labels], columns=[f"pred: {label}" for label in labels])

print(evaluation_note)
display(pd.DataFrame({"class_count": class_counts.reindex(labels)}))
display(metrics)
display(cm_df)

## 6. Train final model for anomaly scoring

Next we train the selected model on all available labeled items and score every item. The score is not a proof that an item is mispriced. It is a way to ask: does this item look like the rarity tier printed in the catalog, compared with the rest of this catalog?

This section also adds an Estimated Strength Score. It is not an official balance or pricing score. It is a model-estimated power signal based only on this catalog. Higher scores mean the item looks mechanically similar to higher-rarity items in this dataset. Lower scores mean the item looks simpler or more situational.

An item is flagged when one or more transparent rules apply:

- The predicted rarity is at least two tiers away from the stated rarity.
- The model assigns a low probability to the stated rarity.
- A low-rarity item has unusually high complexity for its tier.
- A high-rarity item has unusually low complexity for its tier.

In [ ]:
final_model = rarity_model.fit(X, y)
predicted_rarity = final_model.predict(X)
probabilities = final_model.predict_proba(X)
class_labels = list(final_model.named_steps["model"].classes_)
class_to_index = {label: index for index, label in enumerate(class_labels)}

actual_probability = np.array([probabilities[index, class_to_index[actual]] for index, actual in enumerate(y)])
predicted_probability = probabilities.max(axis=1)
class_ranks = np.array([rarity_rank[label] for label in class_labels], dtype=float)
max_rarity_rank = max(rarity_rank.values())
expected_rarity_rank = probabilities @ class_ranks

scored_items = model_df.copy()
scored_items["actual_rarity"] = y.values
scored_items["predicted_rarity"] = predicted_rarity
scored_items["actual_probability"] = actual_probability
scored_items["predicted_probability"] = predicted_probability
scored_items["expected_rarity_rank"] = expected_rarity_rank
scored_items["estimated_strength_score"] = np.round(expected_rarity_rank / max_rarity_rank * 100, 1)
scored_items["listed_rarity_rank"] = scored_items["actual_rarity"].map(rarity_rank)
scored_items["strength_minus_listed"] = np.round(scored_items["expected_rarity_rank"] - scored_items["listed_rarity_rank"], 2)
scored_items["predicted_rarity_rank"] = scored_items["predicted_rarity"].map(rarity_rank)
scored_items["actual_rarity_rank"] = scored_items["listed_rarity_rank"]
scored_items["ordinal_gap"] = scored_items["predicted_rarity_rank"] - scored_items["actual_rarity_rank"]
scored_items["absolute_gap"] = scored_items["ordinal_gap"].abs()

tier_high_complexity = scored_items.groupby("actual_rarity")["complexity_score"].quantile(0.75)
tier_low_complexity = scored_items.groupby("actual_rarity")["complexity_score"].quantile(0.25)
global_high_complexity = scored_items["complexity_score"].quantile(0.75)
global_low_complexity = scored_items["complexity_score"].quantile(0.25)

scored_items["large_prediction_gap"] = scored_items["absolute_gap"] >= 2
scored_items["low_actual_probability"] = scored_items["actual_probability"] <= 0.20
scored_items["low_rarity_high_complexity"] = (
    (scored_items["actual_rarity_rank"] <= 1)
    & (scored_items["complexity_score"] >= scored_items["actual_rarity"].map(tier_high_complexity).fillna(global_high_complexity))
    & (scored_items["complexity_score"] >= global_high_complexity)
)
scored_items["high_rarity_low_complexity"] = (
    (scored_items["actual_rarity_rank"] >= 4)
    & (scored_items["complexity_score"] <= scored_items["actual_rarity"].map(tier_low_complexity).fillna(global_low_complexity))
    & (scored_items["complexity_score"] <= global_low_complexity)
)


def explain_scored_item(row):
    parts = []
    if row["large_prediction_gap"]:
        direction = "higher" if row["ordinal_gap"] > 0 else "lower"
        parts.append(f"the model predicts a {direction} rarity by {int(row['absolute_gap'])} tiers")
    if row["low_actual_probability"]:
        parts.append(f"the stated rarity receives only {row['actual_probability']:.0%} model probability")
    if row["strength_minus_listed"] >= 1.0:
        parts.append(f"the estimated strength is {row['strength_minus_listed']:.2f} tiers above the listing")
    if row["strength_minus_listed"] <= -1.0:
        parts.append(f"the estimated strength is {abs(row['strength_minus_listed']):.2f} tiers below the listing")
    if row["low_rarity_high_complexity"]:
        parts.append("its extracted complexity is high for a low-rarity item")
    if row["high_rarity_low_complexity"]:
        parts.append("its extracted complexity is low for a high-rarity item")
    if not parts:
        parts.append("it is one of the less typical items by model confidence")

    return (
        f"Listed as {row['actual_rarity']} and predicted as {row['predicted_rarity']}; "
        f"{'; '.join(parts)}. "
        f"Strength score {row['estimated_strength_score']:.1f}/100, "
        f"complexity {row['complexity_score']:.2f}, "
        f"effects {int(row['effect_count'])}, usage limits {int(row['usage_limit_count'])}, "
        f"charge pools {int(row['charge_pool_count'])}."
    )


scored_items["reason"] = scored_items.apply(explain_scored_item, axis=1)

top_strength_columns = [
    "name",
    "actual_rarity",
    "predicted_rarity",
    "estimated_strength_score",
    "complexity_score",
    "effect_count",
    "usage_limit_count",
    "charge_pool_count",
    "source_pages",
]
print("Top Estimated Strength Scores")
display(scored_items.sort_values("estimated_strength_score", ascending=False)[top_strength_columns].head(15))

largest_disagreement_columns = [
    "name",
    "actual_rarity",
    "predicted_rarity",
    "estimated_strength_score",
    "strength_minus_listed",
    "ordinal_gap",
    "actual_probability",
    "predicted_probability",
    "reason",
]
print("Largest Rarity Disagreements")
display(
    scored_items.sort_values(["absolute_gap", "actual_probability"], ascending=[False, True])[
        largest_disagreement_columns
    ].head(15)
)


## 7. Explain flagged anomalies

The table below is designed for review. Each row gives the stated rarity, model-predicted rarity, Estimated Strength Score, model confidence, key complexity counts, source pages, and a plain-English reason for the flag.

In [ ]:
pd.set_option("display.max_colwidth", 300)


def compact_text(value, max_chars=280):
    text = " ".join(str(value or "").split())
    return text[:max_chars] + ("..." if len(text) > max_chars else "")


def top_effect_categories(row):
    categories = []
    for column in effect_category_features:
        count = int(row.get(column, 0) or 0)
        if count:
            categories.append((column.replace("effect_category_", ""), count))
    categories = sorted(categories, key=lambda item: item[1], reverse=True)[:3]
    return ", ".join(f"{name} ({count})" for name, count in categories)


def explain_anomaly(row):
    signals = []
    if row["large_prediction_gap"]:
        direction = "higher" if row["ordinal_gap"] > 0 else "lower"
        signals.append(f"the model predicts a {direction} rarity by {int(row['absolute_gap'])} tiers")
    if row["low_actual_probability"]:
        signals.append(f"the stated rarity receives only {row['actual_probability']:.0%} model probability")
    if row["strength_minus_listed"] >= 1.0:
        signals.append(f"the strength estimate is {row['strength_minus_listed']:.2f} tiers above the listed rarity")
    if row["strength_minus_listed"] <= -1.0:
        signals.append(f"the strength estimate is {abs(row['strength_minus_listed']):.2f} tiers below the listed rarity")
    if row["low_rarity_high_complexity"]:
        signals.append("the item is low-rarity but mechanically busy for this catalog")
    if row["high_rarity_low_complexity"]:
        signals.append("the item is high-rarity but mechanically narrow or simple for this catalog")
    if not signals:
        signals.append("the item is less typical than most catalog entries by model confidence")

    mechanics = (
        f"It has {int(row['effect_count'])} extracted effects, "
        f"{int(row['usage_limit_count'])} usage limits, "
        f"{int(row['charge_pool_count'])} charge pools, "
        f"complexity score {row['complexity_score']:.2f}, "
        f"and Estimated Strength Score {row['estimated_strength_score']:.1f}/100."
    )
    return (
        f"Listed as {row['actual_rarity']} and predicted as {row['predicted_rarity']}; "
        f"{'; '.join(signals)}. {mechanics} Review the source text before changing rarity."
    )


anomaly_mask = (
    scored_items["large_prediction_gap"]
    | scored_items["low_actual_probability"]
    | scored_items["low_rarity_high_complexity"]
    | scored_items["high_rarity_low_complexity"]
)

scored_items["top_effect_categories"] = scored_items.apply(top_effect_categories, axis=1)
scored_items["raw_text_preview"] = scored_items["raw_text"].apply(compact_text)
scored_items["reason"] = scored_items.apply(explain_anomaly, axis=1)

anomalies = scored_items.loc[anomaly_mask].copy()
if anomalies.empty:
    anomalies = scored_items.sort_values(["actual_probability", "absolute_gap"], ascending=[True, False]).head(10).copy()
else:
    anomalies = anomalies.sort_values(
        ["absolute_gap", "actual_probability", "complexity_score"],
        ascending=[False, True, False],
    ).head(15)

anomalies = anomalies[
    [
        "name",
        "actual_rarity",
        "predicted_rarity",
        "estimated_strength_score",
        "strength_minus_listed",
        "actual_probability",
        "predicted_probability",
        "ordinal_gap",
        "absolute_gap",
        "complexity_score",
        "effect_count",
        "usage_limit_count",
        "charge_pool_count",
        "source_pages",
        "reason",
        "top_effect_categories",
        "raw_text_preview",
    ]
]

display(anomalies)


## Top Review Candidates

The current top anomaly candidates should be treated as review candidates, not automatic corrections.

1. **Horn of Bronze Dragon Control**: Listed as very rare but predicted uncommon. This may be very powerful in a narrow context because it targets bronze dragons specifically. The model likely sees it as mechanically narrow because its strongest value depends on a specific creature type being present.

2. **Compass of True North**: Listed as rare but predicted legendary. Broad utility effects can make an item look stronger than typical rare items in this catalog, especially when the effect is useful across many travel, planar, or exploration situations.

3. **Amulet of Fiendish Protection**: Listed as very rare but predicted rare. This is a strong defensive item, but its benefit is limited to fiend-related threats, making it appear narrower than other very rare items.


## 8. Visualize anomalies

These charts show how item complexity changes across rarity tiers and where the top review candidates sit. A perfect rarity system would not necessarily produce a straight line, but very high-complexity low-rarity items and very low-complexity high-rarity items deserve a closer look.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    scored_items["actual_rarity_rank"],
    scored_items["complexity_score"],
    c=scored_items["absolute_gap"],
    cmap="viridis",
    alpha=0.75,
)
ax.set_xticks(range(len(rarity_order)))
ax.set_xticklabels(rarity_order, rotation=30, ha="right")
ax.set_xlabel("Stated rarity")
ax.set_ylabel("Complexity score")
ax.set_title("Rarity tier vs. extracted complexity")
fig.colorbar(scatter, ax=ax, label="Prediction tier gap")

for _, row in anomalies.head(8).iterrows():
    x = rarity_rank.get(row["actual_rarity"], 0)
    ax.text(x + 0.03, row["complexity_score"], str(row["name"])[:24], fontsize=8)

plt.tight_layout()
plt.show()

avg_complexity = scored_items.groupby("actual_rarity")["complexity_score"].mean().reindex(rarity_order).dropna()
fig, ax = plt.subplots(figsize=(7, 4))
avg_complexity.plot(kind="bar", ax=ax, color="#e15759")
ax.set_title("Average complexity score by stated rarity")
ax.set_xlabel("Rarity")
ax.set_ylabel("Average complexity score")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted rarity")
ax.set_ylabel("Actual rarity")
ax.set_title("Rarity prediction confusion matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("Top anomaly candidates:")
display(
    anomalies[
        [
            "name",
            "actual_rarity",
            "predicted_rarity",
            "estimated_strength_score",
            "strength_minus_listed",
            "complexity_score",
            "reason",
        ]
    ].head(10)
)


## 9. Conclusion for Reznar

This analysis should be used as a review assistant, not a final judge. The model is useful because it highlights items whose extracted mechanics do not resemble their stated rarity tier, but rarity still depends on design intent, table context, lore, and scarcity.

The top three review candidates from the current run are:

- **Horn of Bronze Dragon Control**: likely appears lower-powered to the model because its strongest effect is narrow and bronze-dragon-specific.
- **Compass of True North**: likely appears higher-powered to the model because broad utility can resemble higher-rarity items in this catalog.
- **Amulet of Fiendish Protection**: likely appears slightly lower than listed because the defense is strong but limited to fiend-related threats.

Estimated Strength Score helps rank items by a model-estimated power signal. It is most useful for prioritizing review: high scores on low-rarity items and low scores on high-rarity items deserve closer inspection.

Main limitations:

- The dataset is small, so model evaluation is approximate.
- Extracted data may contain minor errors from scanned-page OCR or model interpretation.
- Rarity includes subjective, lore, and table-design considerations.
- The model learns from this catalog only, not from a complete official magic item economy.

Recommended next step: inspect the top review candidates against the original PDF pages, then decide whether their rarity labels should change or whether their apparent mismatch is intentional.


## Optional: Export result tables

The notebook does not write files during a normal run. If Reznar wants spreadsheet-friendly review artifacts, uncomment and run the cell below. The CSV files are generated debugging and review outputs under `data/reports/`; they should not be committed.


In [ ]:
# from pathlib import Path
#
# reports_dir = Path("data/reports")
# reports_dir.mkdir(parents=True, exist_ok=True)
#
# anomalies.to_csv(reports_dir / "rarity_anomalies.csv", index=False)
# scored_items.to_csv(reports_dir / "rarity_scored_items.csv", index=False)
#
# print(f"Wrote reports to {reports_dir}")
